In [ ]:
!pip install -q --upgrade langchain langgraph langchain_openai tavily-python amadeus python-dotenv gradio langchain_community graphviz


In [ ]:
import os
import uuid
import getpass
from typing import TypedDict, Annotated, Sequence, List, Tuple, Optional, Any, Union, Literal,  Tuple
import operator
from datetime import date
from IPython.display import display, Markdown, Image
from graphviz import Source
import uuid  

from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.messages import BaseMessage, ToolMessage, HumanMessage, AIMessage, SystemMessage
from langchain.tools import tool
from langchain_core.pydantic_v1 import BaseModel  


from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode  

import gradio as gr

from dotenv import load_dotenv
load_dotenv()

openai_api_key = os.environ["OPENAI_API_KEY"]

print("API Keys loaded (partially hidden for security):")
print(f"OpenAI Key starts with: {openai_api_key[:5]}...")


In [ ]:
def print_markdown(text):
    display(Markdown(text))

In [ ]:
tavily_api_key=os.environ["TAVILY_API_KEY"]
print(f"Tavily Key starts with: {tavily_api_key[:5]}...")

In [ ]:
 llm=ChatOpenAI(temperature=0,openai_api_key=openai_api_key,
          openai_api_base="https://openrouter.ai/api/v1",
          model="mistralai/mistral-small-3.2-24b-instruct:free",
               streaming=True)
 print("LangChain OpenAI Chat Model configured.")

In [ ]:
class AgentState(TypedDict):
    messages:Annotated[Sequence[BaseMessage],operator.add]

In [ ]:
from langchain_community.tools.tavily_search import TavilySearchResults

tavily_search_tool=TavilySearchResults(max_result=3)
tools_list_single=[tavily_search_tool]

In [ ]:
def make_call_models_with_tools(tools:list):
    def call_models_with_tools(state:AgentState):
        messages=state["messages"]
        models_with_tools=llm.bind_tools(tools)
        response=models_with_tools.invoke(messages)
        return{"messages":[response]}

    return call_models_with_tools

In [ ]:
import re
def should_continue(state: AgentState) -> str:
    last_message = state["messages"][-1]
    print("DEBUG: Entering should_continue node")

    # Case 1: Structured tool calls
    if getattr(last_message, "tool_calls", None):
        print("DEBUG: Decision → action (structured tool call)")
        return "action"

    # Case 2: Fallback: check text content for [TOOL_CALL...]
    if isinstance(last_message.content, str) and "[TOOL_CALL" in last_message.content:
        print("DEBUG: Decision → action (text-based tool call detected)")
        return "action"

    # Optional: regex for stricter detection
    if isinstance(last_message.content, str):
        if re.search(r"\[TOOL_CALL.*ARGS\{.*\}\]", last_message.content):
            print("DEBUG: Decision → action (regex detected tool call)")
            return "action"

    # Default → End the workflow
    print("DEBUG: Decision → END (no tool call found)")
    return END

In [ ]:
def build_one_graph_tool(tools_list):
    tool_node=ToolNode(tools_list)
    call_fun_node=make_call_models_with_tools(tools_list)
    graph_one_tool=StateGraph(AgentState)
    graph_one_tool.add_node("agent",call_fun_node)
    graph_one_tool.add_node("action",tool_node)
    graph_one_tool.set_entry_point("agent")
    graph_one_tool.add_conditional_edges(
        "agent",  # Source node name
        should_continue,  # Function to decide the route
        {"action": "action", END: END},  # Mapping: {"decision": "destination_node_name"}
    )
    graph_one_tool.add_edge("action","agent")     
    app = graph_one_tool.compile()
    display(Image(app.get_graph().draw_mermaid_png()))

    return app

    

In [ ]:
def app_call(app, messages):
    # Initialize the state with the provided messages
    initial_state = {"messages": [HumanMessage(content=messages)]}

    # Invoke the app with the initial state
    final_state = app.invoke(initial_state)

    # Iterate through the messages in the final state
    for i in final_state["messages"]:
        # Print the type of the message in markdown format
        print_markdown(i.type)
        # Print the content of the message in markdown format
        print_markdown(i.content)
        # Print any additional kwargs associated with the message
        if i.additional_kwargs != {}:
            print(i.additional_kwargs)

    # Return the content of the last message and the final state
    return final_state["messages"][-1].content, final_state

In [ ]:
app=build_one_graph_tool(tools_list_single)

In [ ]:
messages = "What's the latest news on France in May 2025? Is it a good time to visit?"
output, history = app_call(app, messages)

print("\n==================== OUTPUT ====================")
print(output)

print("\n==================== HISTORY ===================")
print(history)

In [ ]:
amadeus_api_key=os.environ["AMADEUS_API_KEY"]
amadeus_api_secret=os.environ["AMADEUS_API_SECRET"]
print(f"Amadeus Key starts with: {amadeus_api_key[:5]}...")
print(f"Amadeus Secret starts with: {amadeus_api_secret[:5]}...")

In [ ]:
@tool
def get_current_date_tool():
    """Returns the current date in 'YYYY-MM-DD' format. Useful for finding flights/hotels relative to today."""
    return date.today().isoformat()


app_current_date = build_one_graph_tool([get_current_date_tool])

prompt = "What is the current date?"
output, history = app_call(app_current_date, prompt)

In [ ]:
from amadeus import Client, ResponseError
amadeus_client = Client(
    client_id=amadeus_api_key,
    client_secret=amadeus_api_secret,
    hostname = "test",  # Start with the test environment
)

In [ ]:
@tool
def search_flights_tool(
    origin_code: str,
    destination_code: str,
    departure_date: str,
    return_date: str | None = None,
    adults: int = 1,
    travel_class: str = "ECONOMY",
    currency: str = "USD",
    max_offers: int = 5,
):
    """
    Searches live flight prices and availability via Amadeus Flight Offers Search API.
    Required:
        origin_code, destination_code – IATA airport/city codes (e.g., 'YYZ', 'LHR')
        departure_date – 'YYYY-MM-DD'
    Optional:
        return_date – for round‑trips; omit for one‑way
        adults – number of adult passengers (default 1)
        travel_class – 'ECONOMY', 'PREMIUM_ECONOMY', 'BUSINESS', 'FIRST'
        currency – 3‑letter code for pricing (default USD)
        max_offers – how many offers to list back
    """

    print(
        f"DEBUG: Calling Amadeus Flight Search – "
        f"{origin_code}->{destination_code}, "
        f"Depart {departure_date}, Return {return_date}, "
        f"Adults {adults}, Class {travel_class}"
    )

    # --- Call Amadeus Flight Offers Search API ---
    flight_search_params = {
        "originLocationCode": origin_code,
        "destinationLocationCode": destination_code,
        "departureDate": departure_date,
        "adults": adults,
        "travelClass": travel_class,
        "currencyCode": currency,
        "max": max_offers,
    }
    if return_date:
        flight_search_params["returnDate"] = return_date

    response = amadeus_client.shopping.flight_offers_search.get(**flight_search_params)

    # --- Parse the response ---
    if not response.data:
        return (
            f"No flight offers found for {origin_code} → {destination_code} on "
            f"{departure_date}{' (return '+return_date+')' if return_date else ''}."
        )

    results = []
    for offer in response.data[:max_offers]:
        price = offer["price"]["total"]
        airline = offer["validatingAirlineCodes"][0]
        itinerary = offer["itineraries"][0]
        segments = itinerary["segments"]
        first_leg = segments[0]
        last_leg = segments[-1]
        dep_time = first_leg["departure"]["at"][:16].replace("T", " ")
        arr_time = last_leg["arrival"]["at"][:16].replace("T", " ")
        duration = itinerary["duration"].replace("PT", "")
        results.append(f"{airline} | {dep_time} → {arr_time} | {duration} | {price} {currency}")

    return "Found flight options:\n- " + "\n- ".join(results)


# Create Flight SearchTools List
tools_list_full = [
    search_flights_tool,
    get_current_date_tool
]

app_flight_search = build_one_graph_tool(tools_list_full)

In [ ]:
# Prepare your input
prompt = "I want to go to Paris from Toronto for the first week of June. Can you find flight options for 2 adults?"
output, history = app_call(app_flight_search, prompt)

print("\n==================== OUTPUT ====================")
print(output)

print("\n==================== HISTORY ===================")
print(history)

In [ ]:
tools = [
    tavily_search_tool,
    search_flights_tool,
    get_current_date_tool
]

app_travel_agent = build_one_graph_tool(tools)

# Prepare your input
prompt = "I want the latest news about New York, I'm planning to visit from 2025-06-01 to 2025-06-04, leaving from Toronto. Please fetch security and travel advisories, find the cheapest flight for one adult. Finally, format the combined output."
output, history = app_call(app_travel_agent, prompt)